# Asper Beauty – Branch Efficiency Analysis

This notebook provides a **data-driven efficiency pipeline** for an Asper Beauty branch.
It covers:

1. **Data loading** – from a local CSV or a Supabase export
2. **Branch Efficiency Score (BES)** – rank your branch across branches using weighted percentile-based metrics
3. **Demand forecasting** – predict future sales peaks with Prophet
4. **Anomaly detection** – flag unusual sales dips/spikes with PyOD
5. **Customer clustering** – segment customers by purchase behaviour with KMeans
6. **Interactive dashboard** – visualise all results with Plotly

**Setup:**
```bash
pip install -r requirements-notebooks.txt
```
Then set `SUPABASE_URL` and `SUPABASE_ANON_KEY` (or `SUPABASE_SERVICE_ROLE_KEY`, depending on your access level) in your environment (or `.env`) if loading from Supabase, or point `SALES_CSV_PATH` at your exported CSV.

In [ ]:
# Install once (uncomment if needed):
# !pip install numpy scikit-learn xgboost prophet plotly scipy pyod joblib pandas python-dotenv

## 0 · Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # loads .env if present
warnings.filterwarnings("ignore")

# ── Configuration ────────────────────────────────────────────────────────────
# Path to a local branch sales CSV (set to None to use Supabase instead)
SALES_CSV_PATH: str | None = "../data/branch_sales.csv"

# Your branch identifier (used to filter when the CSV contains multiple branches)
BRANCH_ID: str = "branch_01"

# Column name mappings – adjust to match your CSV headers
COL_DATE       = "date"          # ISO date string, e.g. 2024-01-15
COL_BRANCH     = "branch_id"
COL_REVENUE    = "revenue"       # numeric (float)
COL_UNITS      = "units_sold"    # numeric (int)
COL_COST       = "cost"          # numeric (float)
COL_CUSTOMER   = "customer_id"   # string / UUID
COL_CATEGORY   = "category"      # product category string
# ─────────────────────────────────────────────────────────────────────────────

print("Configuration loaded.")

## 1 · Data loading

Load from a local CSV **or** from Supabase. The notebook auto-generates a synthetic
dataset when neither source is available, so you can explore the full pipeline
without real data.

In [ ]:
from joblib import Memory  # noqa: F401  (imported for optional use; remove if not caching)


def _synthetic_data(n_days: int = 365, n_branches: int = 5, seed: int = 42) -> pd.DataFrame:
    """Generate a realistic synthetic sales dataset for demonstration."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2023-01-01", periods=n_days, freq="D")
    categories = ["Skincare", "Makeup", "Haircare", "Fragrance", "Tools"]
    branches = [f"branch_{i:02d}" for i in range(1, n_branches + 1)]

    rows = []
    for branch in branches:
        multiplier = rng.uniform(0.7, 1.3)  # branch-level performance factor
        for date in dates:
            category = rng.choice(categories)
            units = int(rng.poisson(30 * multiplier))
            price = rng.uniform(15, 120)
            cost_ratio = rng.uniform(0.35, 0.55)
            rows.append({
                COL_DATE: date.date().isoformat(),
                COL_BRANCH: branch,
                COL_CATEGORY: category,
                COL_CUSTOMER: f"cust_{rng.integers(1, 200):04d}",
                COL_UNITS: units,
                COL_REVENUE: round(units * price * multiplier, 2),
                COL_COST: round(units * price * multiplier * cost_ratio, 2),
            })

    return pd.DataFrame(rows)


# ── Load data ─────────────────────────────────────────────────────────────────
df_all: pd.DataFrame

if SALES_CSV_PATH and os.path.exists(SALES_CSV_PATH):
    print(f"Loading CSV: {SALES_CSV_PATH}")
    df_all = pd.read_csv(SALES_CSV_PATH, parse_dates=[COL_DATE])

elif os.environ.get("SUPABASE_URL"):
    print("Loading from Supabase …")
    try:
        from supabase import create_client
        sb = create_client(os.environ["SUPABASE_URL"],
                            os.environ.get("SUPABASE_ANON_KEY") or os.environ["SUPABASE_SERVICE_ROLE_KEY"])
        result = sb.table("branch_sales").select("*").execute()
        df_all = pd.DataFrame(result.data)
        df_all[COL_DATE] = pd.to_datetime(df_all[COL_DATE])
        print(f"Loaded {len(df_all):,} rows from Supabase.")
    except Exception as exc:
        print(f"Supabase load failed: {exc}. Falling back to synthetic data.")
        df_all = _synthetic_data()

else:
    print("No data source found – using synthetic data for demonstration.")
    df_all = _synthetic_data()

df_all[COL_DATE] = pd.to_datetime(df_all[COL_DATE])
df_all["profit"] = df_all[COL_REVENUE] - df_all[COL_COST]
df_all["margin"] = (df_all["profit"] / df_all[COL_REVENUE].replace(0, np.nan)).fillna(0)

print(f"Total rows: {len(df_all):,} | Date range: {df_all[COL_DATE].min().date()} → {df_all[COL_DATE].max().date()}")
df_all.head(3)

## 2 · Branch Efficiency Score (BES)

The **BES** is a composite 0–100 score built from three weighted KPIs using
percentile ranks across branches:

| KPI | Weight |
|-----|--------|
| Revenue per day | 40 % |
| Profit margin | 35 % |
| Units sold per day | 25 % |

In [ ]:


def compute_bes(df: pd.DataFrame, branch_col: str, date_col: str) -> pd.DataFrame:
    """Return a DataFrame with BES and component scores for every branch."""
    n_days = df.groupby(branch_col)[date_col].nunique().rename("n_days")
    agg = df.groupby(branch_col).agg(
        total_revenue=(COL_REVENUE, "sum"),
        total_units=(COL_UNITS, "sum"),
        avg_margin=("margin", "mean"),
    ).join(n_days)

    agg["rev_per_day"]   = agg["total_revenue"] / agg["n_days"]
    agg["units_per_day"] = agg["total_units"]   / agg["n_days"]

    # Score each KPI as percentile rank (0–100) across branches using vectorized rank
    for kpi in ["rev_per_day", "avg_margin", "units_per_day"]:
        agg[f"{kpi}_pct"] = agg[kpi].rank(pct=True) * 100

    agg["BES"] = (
        0.40 * agg["rev_per_day_pct"]
        + 0.35 * agg["avg_margin_pct"]
        + 0.25 * agg["units_per_day_pct"]
    ).round(1)

    return agg.sort_values("BES", ascending=False).reset_index()


bes_df = compute_bes(df_all, COL_BRANCH, COL_DATE)

my_bes = bes_df.loc[bes_df[COL_BRANCH] == BRANCH_ID, "BES"]
score  = my_bes.values[0] if not my_bes.empty else None

print(f"\n{'─'*45}")
print(f"  Branch Efficiency Score for '{BRANCH_ID}': {score} / 100")
print(f"{'─'*45}\n")
display(bes_df[[COL_BRANCH, "rev_per_day", "avg_margin", "units_per_day", "BES"]])

## 3 · Demand forecasting with Prophet

Forecast the next **30 days** of daily revenue for the selected branch.

In [ ]:
from prophet import Prophet
import plotly.graph_objects as go

# Prepare Prophet input: requires columns named 'ds' (date) and 'y' (value)
ts = (
    df_all[df_all[COL_BRANCH] == BRANCH_ID]
    .groupby(COL_DATE)[COL_REVENUE]
    .sum()
    .reset_index()
    .rename(columns={COL_DATE: "ds", COL_REVENUE: "y"})
)

if len(ts) < 14:
    print("Not enough history for forecasting (need ≥ 14 days). Using full synthetic dataset.")
    ts = df_all.groupby(COL_DATE)[COL_REVENUE].sum().reset_index().rename(
        columns={COL_DATE: "ds", COL_REVENUE: "y"})

model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
model.fit(ts)

future   = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

# Plot with Plotly
fig_forecast = go.Figure([
    go.Scatter(x=ts["ds"], y=ts["y"],
               name="Actual", mode="lines", line=dict(color="#6366f1")),
    go.Scatter(x=forecast["ds"], y=forecast["yhat"],
               name="Forecast", mode="lines", line=dict(color="#f59e0b", dash="dash")),
    go.Scatter(
        x=pd.concat([forecast["ds"], forecast["ds"].iloc[::-1]]),
        y=pd.concat([forecast["yhat_upper"], forecast["yhat_lower"].iloc[::-1]]),
        fill="toself", fillcolor="rgba(245,158,11,0.15)",
        line=dict(color="rgba(255,255,255,0)"),
        name="Confidence band",
    ),
])
fig_forecast.update_layout(
    title=f"30-day Revenue Forecast – {BRANCH_ID}",
    xaxis_title="Date", yaxis_title="Revenue (SAR)",
    template="plotly_white", legend=dict(orientation="h"),
)
fig_forecast.show()

peak_date = forecast.loc[forecast["ds"] > ts["ds"].max(), "yhat"].idxmax()
print(f"Predicted peak date in next 30 days: {forecast.loc[peak_date, 'ds'].date()}")

## 4 · Anomaly detection with PyOD

Flag days where revenue is unusually high or low using an **Isolation Forest**
outlier detector. Anomalies appear as red markers in the chart.

In [ ]:
from pyod.models.iforest import IForest
import plotly.express as px

daily = (
    df_all[df_all[COL_BRANCH] == BRANCH_ID]
    .groupby(COL_DATE)
    .agg(revenue=(COL_REVENUE, "sum"), units=(COL_UNITS, "sum"))
    .reset_index()
)

if daily.empty:
    print(f"No data for {BRANCH_ID}. Anomaly detection skipped.")
else:
    features = daily[["revenue", "units"]].values
    clf = IForest(contamination=0.05, random_state=42)
    clf.fit(features)
    daily["anomaly"]       = clf.predict(features)        # 1 = outlier
    daily["anomaly_score"] = clf.decision_function(features)
    daily["label"]         = daily["anomaly"].map({0: "Normal", 1: "Anomaly"})

    n_anomalies = daily["anomaly"].sum()
    print(f"Anomalies detected: {n_anomalies} day(s) out of {len(daily)}")
    if n_anomalies:
        display(daily[daily["anomaly"] == 1][[COL_DATE, "revenue", "units", "anomaly_score"]])

    fig_anom = px.scatter(
        daily, x=COL_DATE, y="revenue",
        color="label",
        color_discrete_map={"Normal": "#6366f1", "Anomaly": "#ef4444"},
        size="units",
        title=f"Daily Revenue Anomalies – {BRANCH_ID}",
        labels={COL_DATE: "Date", "revenue": "Revenue (SAR)"},
        template="plotly_white",
    )
    fig_anom.show()

## 5 · Customer segmentation with KMeans

Cluster customers into **4 behavioural segments** using RFM-style features:

| Feature | Meaning |
|---------|---------|
| `frequency` | Number of purchase days |
| `total_revenue` | Total spend |
| `avg_basket` | Average spend per visit |

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px

branch_df = df_all[df_all[COL_BRANCH] == BRANCH_ID].copy()
if branch_df.empty:
    branch_df = df_all.copy()  # fall back to all branches for demo

rfm = branch_df.groupby(COL_CUSTOMER).agg(
    frequency   =(COL_DATE,    "nunique"),
    total_revenue=(COL_REVENUE, "sum"),
).reset_index()
rfm["avg_basket"] = rfm["total_revenue"] / rfm["frequency"]

X = rfm[["frequency", "total_revenue", "avg_basket"]].values
X_scaled = StandardScaler().fit_transform(X)

N_CLUSTERS = 4
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
rfm["segment"] = km.fit_predict(X_scaled).astype(str)

# Friendly segment labels based on centroid order by total_revenue
segment_revenue = rfm.groupby("segment")["total_revenue"].mean().sort_values(ascending=True)
segment_labels = ["At-Risk", "Occasional", "Regular", "VIP"]
centroid_revenue = dict(zip(segment_revenue.index, segment_labels))
rfm["segment_label"] = rfm["segment"].map(centroid_revenue)

print(rfm.groupby("segment_label")[["frequency", "total_revenue", "avg_basket"]].mean().round(2))

fig_cluster = px.scatter(
    rfm, x="frequency", y="total_revenue",
    color="segment_label",
    size="avg_basket",
    hover_data=[COL_CUSTOMER, "avg_basket"],
    title=f"Customer Segments – {BRANCH_ID}",
    labels={"frequency": "Purchase Frequency", "total_revenue": "Total Revenue (SAR)"},
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_cluster.show()

## 6 · Branch Efficiency Dashboard

A single interactive view combining all KPIs across branches.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

bes_sorted = bes_df.sort_values("BES")
highlight  = ["#ef4444" if b == BRANCH_ID else "#6366f1" for b in bes_sorted[COL_BRANCH]]

fig_dash = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Branch Efficiency Score (BES)",
                    "Daily Revenue by Branch",
                    "Average Profit Margin by Branch"),
)

# BES bar
fig_dash.add_trace(
    go.Bar(x=bes_sorted["BES"], y=bes_sorted[COL_BRANCH],
           orientation="h", marker_color=highlight, name="BES"),
    row=1, col=1,
)

# Revenue per day bar
fig_dash.add_trace(
    go.Bar(x=bes_sorted["rev_per_day"], y=bes_sorted[COL_BRANCH],
           orientation="h", marker_color="#10b981", name="Rev/Day"),
    row=1, col=2,
)

# Margin bar
fig_dash.add_trace(
    go.Bar(x=(bes_sorted["avg_margin"] * 100).round(1), y=bes_sorted[COL_BRANCH],
           orientation="h", marker_color="#f59e0b", name="Margin %"),
    row=1, col=3,
)

fig_dash.update_layout(
    title_text=f"Asper Beauty – Branch Efficiency Dashboard  (highlighted: {BRANCH_ID})",
    showlegend=False,
    height=400,
    template="plotly_white",
)
fig_dash.update_xaxes(col=1, range=[0, 100])
fig_dash.update_xaxes(col=3, ticksuffix="%")
fig_dash.show()

## 7 · Summary

Run the cell below to print a concise performance summary for your branch.

In [ ]:
row = bes_df[bes_df[COL_BRANCH] == BRANCH_ID]
if row.empty:
    print(f"Branch '{BRANCH_ID}' not found in data. Check BRANCH_ID above.")
else:
    r = row.iloc[0]
    rank = bes_df[COL_BRANCH].tolist().index(BRANCH_ID) + 1
    total = len(bes_df)
    print("\n" + "═" * 50)
    print(f"  ASPER BEAUTY – Branch Performance Summary")
    print("═" * 50)
    print(f"  Branch        : {BRANCH_ID}")
    print(f"  BES           : {r['BES']} / 100")
    print(f"  Rank          : #{rank} of {total} branches")
    print(f"  Revenue/Day   : SAR {r['rev_per_day']:,.0f}")
    print(f"  Margin        : {r['avg_margin']*100:.1f}%")
    print(f"  Units/Day     : {r['units_per_day']:.1f}")
    print("═" * 50 + "\n")